In [204]:
import pandas as pd
df_clean=pd.read_csv('SMSSpamCollection_cleaned.csv',sep=',',header=0,names=['label','message'],encoding='latin-1')
df_clean.shape


(5169, 2)

In [205]:
df_clean['label'].value_counts()

,count
label,
ham,4516
spam,653


In [206]:
from sklearn.model_selection import train_test_split




In [207]:
x=df_clean['message']
y=df_clean['label']

In [208]:
x_train, x_test,y_train, y_test= train_test_split (x,y,test_size=0.2,random_state=42,stratify=y)

In [209]:
print(len(x_train),len(x_test))

4135 1034


In [210]:
lens=x_train.str.split().str.len()



In [211]:
lens.quantile(0.95)

np.float64(32.29999999999973)

In [212]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [213]:
tokenizer= Tokenizer(num_words=5000,oov_token='<oov>')

In [214]:
tokenizer.fit_on_texts(x_train)

In [215]:
len(tokenizer.word_index)

7950

In [216]:
train_seq =tokenizer.texts_to_sequences(x_train)

In [217]:
train_seq[0]

[1161, 3542, 2, 63, 77, 155, 22, 3, 79, 42]

In [218]:
train_seq=tokenizer.texts_to_sequences(x_train)

In [219]:
test_seq=tokenizer.texts_to_sequences(x_test)

In [220]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [221]:
x_train_pad=pad_sequences(train_seq,maxlen=40,padding='post',truncating='post')

In [222]:
x_test_pad=pad_sequences(test_seq,maxlen=40,padding='post',truncating='post')

In [223]:
x_train_pad.shape

(4135, 40)

In [224]:
x_test_pad.shape

(1034, 40)

In [225]:
y_train_num=y_train.map({'ham':0,'spam':1})

In [226]:
y_test_num=y_test.map({'ham':0,'spam':1})

In [227]:
y_train_num.value_counts()

,count
label,
0,3613
1,522


In [228]:
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout


In [229]:
from tensorflow.keras.models import Sequential


In [230]:
model=Sequential()

In [231]:
model.add(Embedding(5000,64,input_length=40))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [232]:
model.add(LSTM(64))

In [233]:
model.add(Dropout(0.5))

In [234]:
model.add(Dense(1,activation='sigmoid'))

In [235]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [236]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [237]:
history=model.fit(x_train_pad,y_train_num,epochs=5,batch_size=32,validation_split=0.1)

Epoch 1/5
117/117 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.9242 - loss: 0.2264 - val_accuracy: 0.9807 - val_loss: 0.0704
Epoch 2/5
117/117 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.9871 - loss: 0.0554 - val_accuracy: 0.9879 - val_loss: 0.0517
Epoch 3/5
117/117 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.9906 - loss: 0.0326 - val_accuracy: 0.9807 - val_loss: 0.1071
Epoch 4/5
117/117 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.9930 - loss: 0.0359 - val_accuracy: 0.9831 - val_loss: 0.0804
Epoch 5/5
117/117 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9898 - loss: 0.0355 - val_accuracy: 0.9855 - val_loss: 0.0640


In [238]:
y_pred_prob=model.predict(x_test_pad)

33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


In [239]:
y_pred=(y_pred_prob>0.5).astype(int)

In [240]:
y_pred[:10]

array([[0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0]])

In [241]:
from sklearn.metrics import classification_report,confusion_matrix

In [242]:
print(classification_report(y_test_num,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       903
           1       0.92      0.92      0.92       131

    accuracy                           0.98      1034
   macro avg       0.95      0.96      0.95      1034
weighted avg       0.98      0.98      0.98      1034



In [243]:
confusion_matrix(y_test_num,y_pred)

array([[892,  11],
       [ 10, 121]])

In [244]:
pred=y_pred.flatten()

In [245]:
true=y_test_num.values

In [246]:
pred.shape

(1034,)

In [247]:
x_test[(true==0)&(pred==1)]

,message
2615,Hi Chachi tried calling u now unable to reach u .. Pl give me a missed cal once u c tiz msg Kanagu
1201,Hello-/@drivby-:0quit edrunk sorry iff pthis makes no senrd-dnot no how ^ dancce 2 drum n basq!ihave fun 2nhite x ros xxxxxxx
487,Are you free now?can i call now?
3599,"Please protect yourself from e-threats. SIB never asks for sensitive information like Passwords,ATM/SMS PIN thru email. Never share your password with anybody."
1229,"Im in inperialmusic listening2the weirdest track ever byÂleafcutter johnÂ-sounds like insects being molested&someone plumbing,remixed by evil men on acid!"
3742,CERI U REBEL! SWEET DREAMZ ME LITTLE BUDDY!! C YA 2MORO! WHO NEEDS BLOKES
710,S'fine. Anytime. All the best with it.
4416,I (Career Tel) have added u as a contact on INDYAROCKS.COM to send FREE SMS. To remove from phonebook - sms NO to &lt;#&gt;
3247,Hi darlin i cantdo anythingtomorrow as myparents aretaking me outfor a meal. when are u free? Katexxx
3264,"Nowadays people are notixiquating the laxinorficated opportunity for bambling of entropication.... Have you ever oblisingately opted ur books for the masteriastering amplikater of fidalfication? It is very champlaxigating, i think it is atrocious.. Wotz Ur Opinion???? Junna"


In [248]:
x_test[(true==1)&(pred==0)]

,message
2908,"Hi babe its Jordan, how r u? Im home from abroad and lonely, text me back if u wanna chat xxSP visionsms.com Text stop to stopCost 150p 08712400603"
1423,Hi its LUCY Hubby at meetins all day Fri & I will B alone at hotel U fancy cumin over? Pls leave msg 2day 09099726395 Lucy x CallsÂ£1/minMobsmoreLKPOBOX177HP51FL
3949,Missed call alert. These numbers called but left no message. 07008009200
661,"SMS. ac sun0819 posts HELLO:""You seem cool, wanted to say hi. HI!!!"" Stop? Send STOP to 62468"
2303,Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time with a sexy bitch. How about we go slo n hard! Txt XXX SLO(4msgs)
4499,Check Out Choose Your Babe Videos @ sms.shsex.netUN fgkslpoPW fgkslpo
3890,"In The Simpsons Movie released in July 2007 name the band that died at the start of the film? A-Green Day, B-Blue Day, C-Red Day. (Send A, B or C)"
4185,Want explicit SEX in 30 secs? Ring 02073162414 now! Costs 20p/min
1813,Would you like to see my XXX pics they are so hot they were nearly banned in the uk!
3743,ringtoneking 84484


In [249]:
print(classification_report(y_test_num,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       903
           1       0.92      0.92      0.92       131

    accuracy                           0.98      1034
   macro avg       0.95      0.96      0.95      1034
weighted avg       0.98      0.98      0.98      1034



In [250]:
pd.set_option('display.max_colwidth',None)

In [251]:
x_test[(true==1)&(pred==0)]

,message
2908,"Hi babe its Jordan, how r u? Im home from abroad and lonely, text me back if u wanna chat xxSP visionsms.com Text stop to stopCost 150p 08712400603"
1423,Hi its LUCY Hubby at meetins all day Fri & I will B alone at hotel U fancy cumin over? Pls leave msg 2day 09099726395 Lucy x CallsÂ£1/minMobsmoreLKPOBOX177HP51FL
3949,Missed call alert. These numbers called but left no message. 07008009200
661,"SMS. ac sun0819 posts HELLO:""You seem cool, wanted to say hi. HI!!!"" Stop? Send STOP to 62468"
2303,Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time with a sexy bitch. How about we go slo n hard! Txt XXX SLO(4msgs)
4499,Check Out Choose Your Babe Videos @ sms.shsex.netUN fgkslpoPW fgkslpo
3890,"In The Simpsons Movie released in July 2007 name the band that died at the start of the film? A-Green Day, B-Blue Day, C-Red Day. (Send A, B or C)"
4185,Want explicit SEX in 30 secs? Ring 02073162414 now! Costs 20p/min
1813,Would you like to see my XXX pics they are so hot they were nearly banned in the uk!
3743,ringtoneking 84484


In [252]:
comparison=pd.DataFrame({
    'Model':['Naive Bayes','Logistic Regression','LSTM'],
    'Precision':[1.0000,0.9778,0.98],
    'Recall':[0.7405,0.6718,0.86],
    'F1':[0.8509,0.7964,0.92],
    'Accuracy':[0.9671,0.9565,0.98]
})
comparison

,Model,Precision,Recall,F1,Accuracy
0,Naive Bayes,1.0000,0.7405,0.8509,0.9671
1,Logistic Regression,0.9778,0.6718,0.7964,0.9565
2,LSTM,0.9800,0.8600,0.9200,0.9800
